# 04 — Analisis & Visualisasi Akhir
**Tujuan:** Menampilkan semua visualisasi hasil analisis dan menarik insight naratif.

---

In [ ]:
import os
import sys
# Auto-adjust CWD to project root if executed from notebooks directory
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from itertools import combinations
from collections import defaultdict
import sys, os
sys.path.insert(0, 'src')
from graph_builder import build_cooccurrence_graph, compute_centrality, detect_communities

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'positive': '#4CAF50', 'neutral': '#FFC107', 'negative': '#F44336'}
ASPECTS  = ['harga', 'kualitas', 'pengiriman', 'packing', 'pelayanan']

with open('data/processed/absa_results.json', encoding='utf-8') as f:
    absa_results = json.load(f)

G = build_cooccurrence_graph(absa_results)
centrality  = compute_centrality(G)
partition   = detect_communities(G)
print('Data loaded ✓')

## 1. Heatmap Co-occurrence

In [ ]:
matrix = pd.DataFrame(0, index=ASPECTS, columns=ASPECTS)
for entry in absa_results:
    asps = [a for a in entry.get('aspects', []) if a in ASPECTS]
    for a1, a2 in combinations(sorted(set(asps)), 2):
        matrix.at[a1, a2] += 1
        matrix.at[a2, a1] += 1

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 13})
ax.set_title('Heatmap Frekuensi Co-occurrence Antar Aspek', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/cooccurrence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Negative Alignment Heatmap

In [ ]:
neg_matrix = pd.DataFrame(0.0, index=ASPECTS, columns=ASPECTS)
for u, v, d in G.edges(data=True):
    neg_r = d.get('neg_ratio', 0)
    neg_matrix.at[u, v] = neg_r
    neg_matrix.at[v, u] = neg_r

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(neg_matrix, annot=True, fmt='.2f', cmap='Reds',
            linewidths=0.5, linecolor='white', ax=ax,
            vmin=0, vmax=1, annot_kws={'size': 12})
ax.set_title('Neg-Alignment Ratio per Pasangan Aspek\n(seberapa sering kedua aspek sama-sama negatif)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/figures/neg_alignment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPasangan dengan neg_ratio tertinggi:')
neg_flat = []
for u, v, d in G.edges(data=True):
    neg_flat.append({'Pasangan': f'{u} — {v}',
                     'Co-occurrence': d.get('weight',0),
                     'Neg Count': d.get('neg_alignment',0),
                     'Neg Ratio': d.get('neg_ratio',0)})
pd.DataFrame(neg_flat).sort_values('Neg Ratio', ascending=False)

## 3. Dashboard Ringkasan

In [ ]:
from collections import Counter

sent_counts = Counter(e.get('sentiment_label','neutral') for e in absa_results)
freq = Counter(a for e in absa_results for a in e.get('aspects',[]))

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Dashboard: Aspect Co-occurrence Network\nUlasan Tokopedia', 
             fontsize=15, fontweight='bold', y=1.01)

# Panel 1 — Sentimen pie
ax1 = fig.add_subplot(2, 3, 1)
labels = list(sent_counts.keys())
vals   = list(sent_counts.values())
colors = [PALETTE.get(l, '#9E9E9E') for l in labels]
ax1.pie(vals, labels=labels, colors=colors, autopct='%1.1f%%',
        startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax1.set_title('Distribusi Sentimen')

# Panel 2 — Frekuensi aspek
ax2 = fig.add_subplot(2, 3, 2)
asp_names = list(freq.keys()); asp_vals = list(freq.values())
ax2.barh(asp_names, asp_vals, color='#5C6BC0', edgecolor='white')
for i, v in enumerate(asp_vals):
    ax2.text(v+1, i, str(v), va='center', fontsize=9)
ax2.set_title('Frekuensi Aspek'); ax2.set_xlabel('Count')

# Panel 3 — Centrality tabel
ax3 = fig.add_subplot(2, 3, 3)
df_c = pd.DataFrame(centrality).round(3)
ax3.axis('off')
tbl = ax3.table(cellText=df_c.values, rowLabels=df_c.index,
                colLabels=df_c.columns, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
ax3.set_title('Centrality Metrics', fontweight='bold')

# Panel 4 — Network graph
ax4 = fig.add_subplot(2, 3, (4, 5))
import matplotlib.colors as mcolors
pos = nx.spring_layout(G, k=0.8, seed=42)
tab_colors = list(mcolors.TABLEAU_COLORS.values())
nc = [tab_colors[partition[n] % len(tab_colors)] for n in G.nodes()]
ns = [3000 * centrality['degree'][n] + 800 for n in G.nodes()]
ew = [G[u][v]['weight'] for u,v in G.edges()]
max_w = max(ew) if ew else 1
nx.draw_networkx_nodes(G, pos, node_size=ns, node_color=nc, alpha=0.85,
                       edgecolors='white', linewidths=2, ax=ax4)
nx.draw_networkx_edges(G, pos, width=[1+5*(w/max_w) for w in ew],
                       alpha=0.5, edge_color='gray', ax=ax4)
nx.draw_networkx_labels(G, pos, font_size=11, font_weight='bold', ax=ax4)
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u,v): G[u][v]['weight'] for u,v in G.edges()},
                              font_size=8, ax=ax4)
ax4.set_title('Aspect Co-occurrence Network', fontweight='bold')
ax4.axis('off')

# Panel 5 — Statistik ringkasan
ax5 = fig.add_subplot(2, 3, 6)
ax5.axis('off')
stats = [
    ['Total ulasan (sampel)', len(absa_results)],
    ['Ulasan mengandung aspek', len(absa_results)],
    ['Jumlah aspek (node)', G.number_of_nodes()],
    ['Jumlah edge', G.number_of_edges()],
    ['Density', f'{nx.density(G):.2f}'],
    ['Komunitas', len(set(partition.values()))],
    ['Sentimen positif', f"{sent_counts.get('positive',0)/sum(sent_counts.values())*100:.1f}%"],
]
for i, (label, val) in enumerate(stats):
    ax5.text(0, 1 - i*0.13, f'• {label}: ', fontsize=10, ha='left', va='top')
    ax5.text(0.7, 1 - i*0.13, str(val), fontsize=10, ha='left', va='top', fontweight='bold', color='#1565C0')
ax5.set_title('Ringkasan Statistik', fontweight='bold')
ax5.set_xlim(0,1.2); ax5.set_ylim(0,1)

plt.tight_layout()
plt.savefig('results/figures/dashboard_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard tersimpan di results/figures/dashboard_summary.png')

## 4. Insight & Interpretasi

### 4.1 Struktur Jaringan
Graph aspect co-occurrence memiliki **density = 1.0**, artinya semua aspek saling terhubung. Ini mengindikasikan bahwa ulasan produk di Tokopedia cenderung menyebut multiple aspek sekaligus dalam satu review.

### 4.2 Aspek Paling Dominan
- **Pengiriman** & **Kualitas** adalah pasangan yang paling sering disebut bersamaan. Pembeli Tokopedia sangat peduli pada kondisi barang saat diterima.
- **Pelayanan** sering dikaitkan dengan pengiriman, menunjukkan bahwa penilaian seller turut dipengaruhi oleh performa kurir.

### 4.3 Neg-Alignment
Pasangan aspek dengan neg_ratio tertinggi mengindikasikan aspek mana yang paling sering "gagal bersama" — insight penting untuk seller dalam memprioritaskan perbaikan.

### 4.4 Keterbatasan
- Analisis menggunakan sampel 500 review (random). Hasil bisa bergeser dengan sampel lebih besar.
- Model ABSA dilatih pada domain makanan; validasi lanjutan diperlukan untuk domain marketplace umum.

In [ ]:
import glob
print('=== FILES DIHASILKAN ===')
for f in sorted(glob.glob('results/**/*', recursive=True)):
    if os.path.isfile(f):
        size = os.path.getsize(f)
        print(f'  {f.replace("../", ""):50s}  ({size/1024:.1f} KB)')